# 03 - Modeling and evaluation

Train baseline and optimized models, evaluate performance, and analyze feature importance with SHAP.

In [ ]:
import joblib
import numpy as np
import optuna
import pandas as pd
import shap
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from xgboost import XGBRegressor

from backlink_pricing_model.core.environment import get_project_root
from backlink_pricing_model.core.notebook import init_notebook
from backlink_pricing_model.visualization.importance_plots import (
    plot_feature_importance,
)
from backlink_pricing_model.visualization.models_plots import (
    plot_model_comparison,
    plot_predictions_vs_actuals,
    plot_residuals,
)

init_notebook()

# Suppress Optuna logs during trials.
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Load engineered train/test splits

In [ ]:
root = get_project_root()
train_df = pd.read_parquet(root / "data" / "engineered" / "train_df.parquet")
test_df = pd.read_parquet(root / "data" / "engineered" / "test_df.parquet")

TARGET = "log_price"
FEATURE_COLS = [c for c in train_df.columns if c != TARGET]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Features: {len(FEATURE_COLS)}")

## 2. Evaluation helper

In [ ]:
def evaluate_model(
    name: str, y_true: np.ndarray, y_pred: np.ndarray
) -> dict[str, float]:
    """Compute regression metrics and print a summary."""
    metrics = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
        "r2": r2_score(y_true, y_pred),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
    }
    print(
        f"{name:25s} | MAE: {metrics['mae']:.4f} | "
        f"RMSE: {metrics['rmse']:.4f} | R2: {metrics['r2']:.4f} | "
        f"MAPE: {metrics['mape']:.4f}"
    )
    return metrics

## 3. Baseline models

Train simple baselines to establish a performance floor.

In [ ]:
all_metrics: dict[str, dict[str, float]] = {}

# Baseline 1: Mean prediction.
mean_pred = np.full_like(y_test, y_train.mean())
all_metrics["Mean baseline"] = evaluate_model("Mean baseline", y_test.values, mean_pred)

# Baseline 2: Ridge regression.
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
all_metrics["Ridge"] = evaluate_model("Ridge", y_test.values, ridge_pred)

# Baseline 3: Random Forest.
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
all_metrics["Random Forest"] = evaluate_model("Random Forest", y_test.values, rf_pred)

# Baseline 4: Gradient Boosting.
gb = GradientBoostingRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
all_metrics["Gradient Boosting"] = evaluate_model("Gradient Boosting", y_test.values, gb_pred)

## 4. XGBoost with Optuna hyperparameter optimization

In [ ]:
def xgb_objective(trial: optuna.Trial) -> float:
    """Optuna objective for XGBoost hyperparameter tuning."""
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "n_jobs": -1,
    }
    model = XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False,
    )
    pred = model.predict(X_test)
    return mean_squared_error(y_test, pred, squared=False)


study = optuna.create_study(direction="minimize", study_name="xgb_backlink_pricing")
study.optimize(xgb_objective, n_trials=100, show_progress_bar=True)

print(f"\nBest RMSE: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

## 5. Train final XGBoost model with best params

In [ ]:
best_xgb = XGBRegressor(**study.best_params, random_state=42, n_jobs=-1)
best_xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_pred = best_xgb.predict(X_test)
all_metrics["XGBoost (tuned)"] = evaluate_model("XGBoost (tuned)", y_test.values, xgb_pred)

## 6. Model comparison

In [ ]:
plot_model_comparison(all_metrics, metric_name="rmse").show()
plot_model_comparison(all_metrics, metric_name="r2").show()

## 7. Predictions vs actuals and residual analysis

In [ ]:
plot_predictions_vs_actuals(y_test.values, xgb_pred, title="XGBoost (tuned): predictions vs actuals").show()
plot_residuals(y_test.values, xgb_pred, title="XGBoost (tuned): residual analysis").show()

## 8. Feature importance

In [ ]:
plot_feature_importance(
    FEATURE_COLS,
    best_xgb.feature_importances_,
    title="XGBoost feature importance (gain)",
).show()

## 9. SHAP analysis

In [ ]:
# SHAP TreeExplainer for XGBoost.
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test)

# Summary plot (beeswarm).
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, show=False)

# Bar plot of mean absolute SHAP values.
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, plot_type="bar", show=False)

## 10. Save best model

In [ ]:
models_dir = root / "models"
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "xgb_backlink_pricing.joblib"
joblib.dump(best_xgb, model_path)
print(f"Saved model to {model_path}")

# Save metrics summary.
metrics_df = pd.DataFrame(all_metrics).T
metrics_df.index.name = "model"
metrics_df.to_csv(models_dir / "metrics_summary.csv")
print(f"\nFinal metrics:\n{metrics_df.round(4)}")